# Can-annular (cannular) combustor with its secondary-air system

![Network topology](can_annular_combustor_topology.svg)

A whole combustor, not just one flame tube.
Compressor-discharge air enters a common casing **plenum** and is distributed by the network to a **ring of eight reacting cans**, each burning **Jet-A(L)**; the cans are cross-linked by flame-tube **interconnectors** and the products are collected and accelerated through a **choked nozzle-guide-vane (NGV) throat** on the way to the turbine.

Per can the flow reads left to right: swirl air through the dome, fuel injection, an equilibrium flame, then dilution and cooling air added along the liner, tapering to the exit.
This is the reacting successor to the cold `gas_turbine_large` secondary-air showcase: the same idea of one delivery flow metered across many circuits, now with combustion and a choked exit.

What this case exercises:

* **Reacting mean flow at scale**: eight equilibrium Jet-A(L)/air flames solved together with the air distribution, ~60 elements, in one Newton solve.
* **A choked exit that sets the operating pressure**: with the delivered air fixed, the choked NGV throat area fixes the combustor pressure.
* **A non-tree topology**: the interconnector ring links adjacent cans, so staged fuelling drives emergent can-to-can cross-flow (the reacting analogue of the cold showcase's cross-bridge).
* **A solved secondary-air split**: dome (swirl) air, a turbine-cooling bypass, and the dilution and cooling air are all metered off one plenum through swirler and liner-hole losses, so the network computes the air distribution rather than it being prescribed.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.thermo import SpeciesLibrary
from nefes.thermo.api import EQ_FROZEN, EQ_KERNEL
from nefes.chem.composition import species_mass_fractions, enthalpy_mass, equivalence_ratio_mixture
from nefes.elements import catalog as cat
from nefes.plotting import use_nefes_theme, COLORWAY

use_nefes_theme()  # register + activate the bundled Nefes theme

# A compact Jet-A(L)/air library from the NASA-CEA `thermo.inp`: air species and the
# liquid Jet-A fuel, plus the major products the equilibrium solver fills in.
LIB = SpeciesLibrary.from_cea(species=["Jet-A(L)", "O2", "N2", "Ar", "CO2", "H2O", "CO", "H2", "OH", "O", "H", "NO"])
AIR = {"O2": 0.2095, "N2": 0.7808, "Ar": 0.0093, "CO2": 0.0004}
FUEL = {"Jet-A(L)": 1.0}
H_AIR = enthalpy_mass(LIB, species_mass_fractions(LIB, AIR, "mole"), 300.0)

# stoichiometric Jet-A/air mass ratio, straight from the library
_st = equivalence_ratio_mixture(LIB, FUEL, AIR, 1.0, basis="mass")
F_STOICH = _st["Jet-A(L)"] / (1.0 - _st["Jet-A(L)"])
print("elements:", list(LIB.elements))
print(f"stoichiometric Jet-A/air mass ratio f_stoich = {F_STOICH:.4f}")

## The combustor as a network

One `mass_flow_inlet` delivers the compressor-discharge air into the casing `plenum` (a static-pressure `junction`), and **every** downstream flow is split off it: there is a single air source and the network solves how it distributes.
A **turbine-cooling bypass** leaves through a loss to its own offtake; each can draws **dome/swirl air** through a swirler loss; and the **dilution** and **cooling** air is metered through liner holes (a loss per hole) and mixed into the flame-tube gas at a `junction`, exactly as air enters through the secondary-air and dilution holes in the casing.
So the dome/dilution/cooling split is not prescribed: it is set by the hole and swirler resistances and recovered from the solve.
Fuel is injected just after the swirler (from the fuel manifold, so it stays a `mass_source`); an `equilibrium_flame` burns the rich primary mixture; the dilution and cooling air then mix in along the liner, tapering to the can exit; all eight exits meet at the `ngv_collector` and pass through the `choked_nozzle_outlet` throat.

The closures are pinned per edge: **frozen** up to and across the flame approach and on every fresh-air branch, **equilibrium** from the flame onward and through the mixing junctions, where the added air completes combustion and dilutes the burnt gas.

In [ ]:
F, K = EQ_FROZEN, EQ_KERNEL

# -- operating point --------------------------------------------------------
N_CAN = 8
P3 = 15.0e5  # compressor-discharge pressure scale [Pa]
T3 = 720.0  # compressor-discharge / cooling-air total temperature [K]
MDOT_AIR = 9.0  # total delivered (plenum) air [kg/s]
PHI_OVERALL = 0.44  # nominal overall equivalence ratio (sizes the fuel; the actual split is emergent)
BYPASS_FRAC_NOM = 0.10  # nominal bypass share, used only to size the fuel

# areas [m^2]
A_DIFF = A_PLENUM = 0.060
A_DOME = 0.004  # swirler / dome feed
A_PRIMARY = 0.010
A_LINER = 0.012
A_DIL = 0.004  # dilution-hole feed area
A_COOL = 0.003  # cooling-hole feed area
A_EXIT = 0.008
A_INTERCONN = 0.0015
A_NGV_APPROACH = 0.040
A_NGV_THROAT = 0.010
A_BYPASS = 0.008

# loss coefficients that meter the plenum-to-liner air split (the network solves the split)
K_SWIRL, K_DIL, K_COOL = 10.0, 5.0, 12.0
K_INTERCONN, K_BYPASS = 3.0, 2.5

In [ ]:
def build(n=N_CAN, fuel_stage=None, throat_area=A_NGV_THROAT, phi_overall=PHI_OVERALL):
    """Assemble the can-annular combustor as a Nefes ``Network``.

    ``fuel_stage`` is a per-can multiplier on the (equal) design fuel flow, renormalized to
    hold the total fuel fixed; a non-uniform pattern stages the ring and drives interconnector
    cross-flow.  Returns ``(net, idx, can)`` where ``idx`` names the shared elements and
    ``can[i]`` maps each can's element roles to indices, so edges are recovered by
    ``net.edge_between(tail, head)`` without hand-tracking indices.
    """
    # size the fuel for a nominal overall phi; the actual air split (and so the real phi) is
    # solved by the network from the hole/swirler resistances, not prescribed here.
    fuel_per_can = MDOT_AIR * (1.0 - BYPASS_FRAC_NOM) * F_STOICH * phi_overall / n

    if fuel_stage is None:
        fuel_stage = np.ones(n)
    fuel_stage = np.asarray(fuel_stage, float)
    fuel_stage *= n / fuel_stage.sum()  # hold the total fuel fixed

    nodes = [
        cat.mass_flow_inlet(MDOT_AIR, T3, composition=AIR, name="compressor_discharge"),  # 0
        cat.duct(0.4, name="diffuser"),  # 1
        cat.junction(name="plenum"),  # 2
    ]
    idx = {"inlet": 0, "diffuser": 1, "plenum": 2}
    nxt = 3

    def push(node, key):
        nonlocal nxt
        nodes.append(node)
        here = nxt
        nxt += 1
        if key is not None:
            idx[key] = here
        return here

    # turbine-cooling bypass: plenum -> loss -> offtake sink (below the plenum pressure)
    push(cat.loss(K_BYPASS, name="bypass_loss"), "bypass_loss")
    push(cat.pressure_outlet(P3 * 0.62, Tt_backflow=T3, composition=AIR, name="turbine_cooling"), "bypass_sink")
    # exit collector + choked NGV
    push(cat.junction(name="ngv_collector"), "collector")
    push(cat.choked_nozzle_outlet(throat_area, name="ngv_throat"), "ngv")

    can = []
    for i in range(n):
        d = {}
        d["swirl"] = push(cat.loss(K_SWIRL, name=f"swirl_{i}"), None)
        d["fuel"] = push(cat.mass_source(fuel_per_can * fuel_stage[i], T3, composition=FUEL, name=f"fuel_{i}"), None)
        d["flame"] = push(cat.equilibrium_flame(name=f"flame_{i}"), None)
        d["pz"] = push(cat.junction(name=f"primary_{i}"), None)
        d["dil_hole"] = push(cat.loss(K_DIL, name=f"dil_hole_{i}"), None)  # meters dilution air
        d["dil_mix"] = push(cat.junction(name=f"dilution_{i}"), None)  # mixes it into the burnt gas
        d["cool_hole"] = push(cat.loss(K_COOL, name=f"cool_hole_{i}"), None)
        d["cool_mix"] = push(cat.junction(name=f"cooling_{i}"), None)
        d["taper"] = push(cat.isentropic_area_change(name=f"taper_{i}"), None)
        can.append(d)

    edges, models = [], []

    def add(t, h, a, m):
        edges.append((t, h, a))
        models.append(m)

    add(idx["inlet"], idx["diffuser"], A_DIFF, F)
    add(idx["diffuser"], idx["plenum"], A_PLENUM, F)
    add(idx["plenum"], idx["bypass_loss"], A_BYPASS, F)
    add(idx["bypass_loss"], idx["bypass_sink"], A_BYPASS, F)
    for i in range(n):
        d = can[i]
        # main flame-tube line: dome air -> flame -> primary -> dilution mix -> cooling mix -> exit
        add(idx["plenum"], d["swirl"], A_DOME, F)  # dome/swirl air split off the plenum
        add(d["swirl"], d["fuel"], A_PRIMARY, F)  # swirler loss -> primary
        add(d["fuel"], d["flame"], A_PRIMARY, F)  # fuel injection; flame approach edge
        add(d["flame"], d["pz"], A_PRIMARY, K)  # burnt -> primary junction
        add(d["pz"], d["dil_mix"], A_LINER, K)  # burnt gas into the dilution junction
        add(d["dil_mix"], d["cool_mix"], A_LINER, K)  # -> cooling junction
        add(d["cool_mix"], d["taper"], A_LINER, K)  # -> exit taper
        add(d["taper"], idx["collector"], A_EXIT, K)  # into the NGV collector
        # dilution and cooling air metered off the plenum through liner holes, mixed into the gas
        add(idx["plenum"], d["dil_hole"], A_DIL, F)  # plenum -> dilution hole
        add(d["dil_hole"], d["dil_mix"], A_DIL, F)  # fresh air into the dilution junction
        add(idx["plenum"], d["cool_hole"], A_COOL, F)  # plenum -> cooling hole
        add(d["cool_hole"], d["cool_mix"], A_COOL, F)  # fresh air into the cooling junction
    for i in range(n):  # interconnector ring: adjacent primary junctions
        add(can[i]["pz"], can[(i + 1) % n]["pz"], A_INTERCONN, K)
    add(idx["collector"], idx["ngv"], A_NGV_APPROACH, K)

    net = nefes.Network(
        gas=nefes.equilibrium(LIB),
        p_ref=P3,
        T_ref=T3,
        mdot_ref=MDOT_AIR,
        h_ref=abs(H_AIR),
        nodes=nodes,
        edges=edges,
        edge_models=models,
    )
    return net, idx, can


net, idx, can = build()
sol = net.solve()

mdot = sol.field("mdot")
flame0 = net.edge_between(can[0]["flame"], can[0]["pz"])  # can-0 flame-exit edge
dome = sum(mdot[net.edge_between(idx["plenum"], can[i]["swirl"])] for i in range(N_CAN))
comb = dome + sum(
    mdot[net.edge_between(idx["plenum"], can[i][h])] for h in ("dil_hole", "cool_hole") for i in range(N_CAN)
)
fuel_total = MDOT_AIR * (1.0 - BYPASS_FRAC_NOM) * F_STOICH * PHI_OVERALL
print("converged :", sol.converged, "  Newton iterations:", sol.iterations)
print("elements  :", len(net._elements), "  edges:", sol.field("M").size)
print(f"primary phi = {(fuel_total / dome) / F_STOICH:.2f}   flame T = {sol.field('T')[flame0]:.0f} K")
print(f"turbine-inlet (NGV) T = {sol.field('T')[-1]:.0f} K    max Mach = {sol.field('M').max():.3f}")

## The network, coloured by temperature

The diagram lays the elements out in their physical arrangement: the delivery line on the left, the ring of eight cans stacked in the middle, and the choked NGV on the right.
We stage the fuel here (two rich pilot cans, two turned-down cans) so the temperature spread across the ring is visible and the interconnectors carry flow; the faint vertical arrows at the `primary` column are the interconnector ring.
Edge colour is static temperature, arrow width is mass flow.

`net.plot` accepts explicit `positions` (an element-index to `(x, y)` map), so the ring is drawn in its physical arrangement rather than the default left-to-right layering.

In [ ]:
def layout(idx, can, n=N_CAN):
    """Element-index -> (x, y): delivery line -> ring of cans -> NGV, with the dilution and
    cooling holes tapping the plenum just above each mixing junction."""
    dy = 1.6
    rows = [(n - 1) / 2.0 - i for i in range(n)]
    pos = {
        idx["inlet"]: (0.0, 0.0),
        idx["diffuser"]: (1.0, 0.0),
        idx["plenum"]: (2.0, 0.0),
        idx["bypass_loss"]: (3.0, (n / 2.0 + 1.4) * dy),
        idx["bypass_sink"]: (4.5, (n / 2.0 + 1.4) * dy),
        idx["collector"]: (11.5, 0.0),
        idx["ngv"]: (12.6, 0.0),
    }
    main = {"swirl": 3.0, "fuel": 4.0, "flame": 5.0, "pz": 6.2, "dil_mix": 7.6, "cool_mix": 8.9, "taper": 10.0}
    for i in range(n):
        y = rows[i] * dy
        for key, cx in main.items():
            pos[can[i][key]] = (cx, y)
        pos[can[i]["dil_hole"]] = (7.6, y + 0.55)  # dilution hole tapping the plenum
        pos[can[i]["cool_hole"]] = (8.9, y + 0.55)  # cooling hole
    return pos


# two rich pilot cans (0, 7) and two turned-down cans (3, 4): a staged / part-load ring
stage = np.array([1.35, 1.0, 1.0, 0.65, 0.65, 1.0, 1.0, 1.35])
net_s, idx_s, can_s = build(fuel_stage=stage)
sol_s = net_s.solve()

net_s.plot(
    solution=sol_s,
    color_by="T",
    width_by="mdot",
    colorscale="Inferno",
    positions=layout(idx_s, can_s),
    show_edge_labels=False,
    title="Can-annular combustor (staged fuelling): static temperature [K]",
    width=1400,
    height=900,
)

## Where the delivered air goes

The natural output of a secondary-air model is *how the delivery flow splits* -- and here that split is solved, not assumed.
Every branch is metered off the one `plenum`, so the whole air budget closes against the single inlet: bypass plus dome plus dilution plus cooling equals the delivered air.
The bars below read each share straight off its plenum branch with `net.edge_between`.

In [ ]:
mdot = sol.field("mdot")


# every share is a plenum branch flow, recovered from the element maps
def plenum_total(role):
    return sum(mdot[net.edge_between(idx["plenum"], can[i][role])] for i in range(N_CAN))


bypass = mdot[net.edge_between(idx["plenum"], idx["bypass_loss"])]
dome_total = plenum_total("swirl")
dil_total = plenum_total("dil_hole")
cool_total = plenum_total("cool_hole")
fuel_total = MDOT_AIR * (1.0 - BYPASS_FRAC_NOM) * F_STOICH * PHI_OVERALL
delivered = bypass + dome_total + dil_total + cool_total  # closes against the inlet

labels = ["turbine-cooling<br>bypass", "dome / swirl<br>(8 cans)", "dilution<br>(8 cans)", "cooling<br>(8 cans)"]
vals = [bypass, dome_total, dil_total, cool_total]
colors = [COLORWAY[0], COLORWAY[1], COLORWAY[2], COLORWAY[3]]

fig = go.Figure()
fig.add_bar(x=labels, y=vals, marker_color=colors, text=[f"{v:.2f} kg/s" for v in vals], textposition="outside")
fig.update_layout(
    title=f"Solved air split: {delivered:.2f} kg/s (= {MDOT_AIR:.1f} delivered) + {fuel_total:.2f} kg/s Jet-A",
    yaxis_title="mass flow  [kg/s]",
    showlegend=False,
    height=440,
)
fig.update_yaxes(rangemode="tozero")
fig

## Axial profile through one can

Following a single can from the dome to the collector: the flame jumps to its rich, oxygen-limited equilibrium (hot, rich in CO and H$_2$); the dilution air then completes combustion and cools the gas; the cooling film trims it further to the turbine-inlet value.

In [ ]:
c0 = can[0]
# consecutive edges along can 0, recovered by their (tail, head) element pair
seq = [
    ("swirl", "fuel"),
    ("fuel", "flame"),
    ("flame", "pz"),
    ("pz", "dil_mix"),
    ("dil_mix", "cool_mix"),
    ("cool_mix", "taper"),
]
stations = ["dome air", "+ fuel", "flame", "primary", "+ dilution", "+ cooling"]
eidx = [net.edge_between(c0[a], c0[b]) for a, b in seq]

T = sol.field("T")
x_ax = list(range(len(seq)))
keys = ["CO", "CO2", "O2", "H2O", "H2"]
palette = {"CO": COLORWAY[4], "CO2": COLORWAY[2], "O2": COLORWAY[0], "H2O": COLORWAY[1], "H2": COLORWAY[3]}
prof = {k: [sol.species(e, "mole").get(k, 0.0) for e in eidx] for k in keys}

fig = make_subplots(specs=[[{"secondary_y": True}]])
for k in keys:
    fig.add_scatter(x=x_ax, y=prof[k], name=k, mode="lines+markers", line=dict(color=palette[k]))
fig.add_scatter(
    x=x_ax,
    y=[T[e] for e in eidx],
    name="T",
    mode="lines+markers",
    secondary_y=True,
    line=dict(color="black", dash="dash", width=3),
)
fig.update_xaxes(tickvals=x_ax, ticktext=stations)
fig.update_yaxes(title_text="mole fraction", secondary_y=False)
fig.update_yaxes(title_text="static temperature [K]", secondary_y=True)
fig.update_layout(title="Axial profile through can 0: flame, dilution, cooling", height=460)
fig

## The choked NGV sets the combustor pressure

The nozzle-guide-vane throat runs **choked**: the flow reaches Mach 1 at the throat, so the throat area fixes the corrected mass flow leaving the combustor.
With the delivered air held fixed, shrinking the throat backs the pressure up through the whole combustor; opening it lets the pressure fall and pulls a larger share of the delivery flow through the cans (and less through the bypass).
This is the element that ties the combustor pressure to the turbine that follows it.

In [ ]:
throats = np.array([0.008, 0.009, 0.010, 0.011, 0.012])
e_bypass = net.edge_between(idx["plenum"], idx["bypass_loss"])  # same edge id for every build
p_plenum, ngv_share = [], []
prev = None
for At in throats:
    s = build(throat_area=float(At))[0].solve(x0=prev.x if prev else None)
    prev = s
    m = s.field("mdot")
    p_plenum.append(s.field("p")[1] / 1e5)  # diffuser -> plenum edge
    ngv_share.append(100.0 * m[-1] / (m[-1] + m[e_bypass]))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(
    x=throats * 1e4, y=p_plenum, name="plenum pressure", mode="lines+markers", line=dict(color=COLORWAY[4], width=3)
)
fig.add_scatter(
    x=throats * 1e4,
    y=ngv_share,
    name="NGV share of delivery",
    mode="lines+markers",
    secondary_y=True,
    line=dict(color=COLORWAY[0], width=3, dash="dash"),
)
fig.update_xaxes(title_text="NGV throat area  [cm^2]")
fig.update_yaxes(title_text="plenum static pressure  [bar]", secondary_y=False)
fig.update_yaxes(title_text="NGV share of delivered air  [%]", secondary_y=True)
fig.update_layout(title="Choked NGV: throat area sets the combustor pressure", height=460)
fig

## Staged fuelling and the interconnector ring

The interconnectors make the graph a **ring, not a tree**.
When the ring is fuelled uniformly the cans sit at equal pressure and the interconnectors carry essentially nothing.
Stage the fuel (rich pilots, turned-down neighbours) and the hotter cans sit at slightly higher pressure, so gas flows through the interconnectors from the rich cans toward the lean ones: an emergent cross-flow the solver finds from the topology, with no edge told which way to run.

In [ ]:
m_s = sol_s.field("mdot")
T_s = sol_s.field("T")

ring = [(net_s.edge_between(can_s[i]["pz"], can_s[(i + 1) % N_CAN]["pz"]), i, (i + 1) % N_CAN) for i in range(N_CAN)]
ring_labels = [f"{i}->{j}" for (_e, i, j) in ring]
ring_mdot = [m_s[e] for (e, _i, _j) in ring]
flame_T = [T_s[net_s.edge_between(can_s[i]["flame"], can_s[i]["pz"])] for i in range(N_CAN)]

fig = make_subplots(rows=1, cols=2, subplot_titles=("per-can flame temperature", "interconnector cross-flow"))
fig.add_bar(x=[f"can {i}" for i in range(N_CAN)], y=flame_T, marker_color=COLORWAY[4], row=1, col=1)
fig.add_bar(
    x=ring_labels, y=ring_mdot, marker_color=[COLORWAY[0] if v >= 0 else COLORWAY[2] for v in ring_mdot], row=1, col=2
)
fig.add_hline(y=0.0, line_color="gray", line_dash="dot", row=1, col=2)
fig.update_yaxes(title_text="flame T [K]", row=1, col=1)
fig.update_yaxes(title_text="mass flow [kg/s]", row=1, col=2)
fig.update_layout(title="Staged ring: hot pilots drive flow through the interconnectors", showlegend=False, height=440)
fig

## Turbine-inlet temperature control

The overall (lean) equivalence ratio sets the turbine-inlet temperature the NGV delivers.
Richening the whole ring raises it toward the metallurgical limit; the solver warm-starts each point from the last, so the sweep is a handful of Newton steps apiece.

In [ ]:
phis = np.linspace(0.40, 0.62, 12)
T4 = []
prev = None
for phi in phis:
    s = build(phi_overall=float(phi))[0].solve(x0=prev.x if prev else None)
    prev = s
    T4.append(s.field("T")[-1])

fig = go.Figure()
fig.add_scatter(x=phis, y=T4, mode="lines+markers", line=dict(color=COLORWAY[4], width=3))
fig.update_layout(
    title="Turbine-inlet temperature vs overall equivalence ratio",
    xaxis_title="overall equivalence ratio  phi",
    yaxis_title="turbine-inlet (NGV) temperature  [K]",
    showlegend=False,
    height=440,
)
fig

## Export for Nemo

The full network is available as `net` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML: `sol.to_yaml(path)` embeds the mean-flow result fields, `net.to_yaml(path)` writes the topology only; then open the file in Nemo.
The topology of this case ships alongside the notebook as `can_annular_combustor.yaml`.

In [ ]:
import os, tempfile

_out = os.path.join(tempfile.mkdtemp(), "can_annular_combustor.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use net.to_yaml(_out) for topology only
print("saved case:", _out)